# Can a minimal age × temporal attention bias learn a known interaction?

**Research question.** Can a one-layer Transformer recover

$$b(a,\tau)=-\bigl(\lambda_0+\beta\,z(a)\bigr)\tau$$

when that age × lag interaction is **required** to solve the task?

This notebook is a controlled diagnostic. It does **not** use Fourier age encoding, Chebyshev $T_1\ldots T_5$, $\Delta\alpha(a)$, auxiliary interaction losses, or the production DKM implementation.

| | |
|---|---|
| Code | `age_temporal_interaction_exp/` |
| Raw tables | `age_temporal_interaction_exp/results/*.csv` / `*.json` |
| Figures | `age_temporal_interaction_exp/results/figures/` |
| Matched-pair follow-up | `age_temporal_interaction_exp/results/matched/` |


## 1. Experiment rationale

The previous S4 / DKM run on Synthea did **not** recover a planted age-dependent decay: learned $\lambda(a)$ stayed flat, and polarity counts already solved most of the label. That experiment used a long $\tau$ horizon, softplus + MLP age encoding, age at the prediction head, noisy labels, and mean pooling.

This diagnostic asks a narrower question with a simpler, identifiable mechanism:

1. Plant $y=1[\sum_j w_j x_j>0]$ with $w=\mathrm{softmax}(-\lambda(a)\tau)$ and **no label noise**.
2. Put age into the model **only** as a linear slope on lag (primary arm).
3. Ask whether ordinary BCE recovers the sign of $\beta$ and whether correct age beats shuffled age **only when an interaction exists**.


## 2. Mathematical formulation

Standardize index age on the **training split only**:

$$z(a)=\frac{a-\mu_{\mathrm{train}}}{\sigma_{\mathrm{train}}}$$

Elapsed time uses a fixed synthetic horizon, **not** the MIMIC / 18-year $\tau_{\max}$:

$$\tau=\mathrm{clip}\left(\frac{\log(1+\Delta t_{\mathrm{days}}/7)}{\log(1+90/7)},\,0,1\right)$$

Planted slope and labels (same injected events for T0/T1/T2):

$$\lambda^*(a)=\lambda_0+\beta^* z(a),\qquad
w^*_j=\mathrm{softmax}_j\bigl(-\lambda^*(a)\,\tau_j\bigr),\qquad
r=\sum_j w^*_j x_j,\qquad
y=1[r>0]$$

with $\lambda_0=0.5$ and $\beta^*\in\{0,+1,-1\}$. $\lambda_0$ is smaller than the illustrative value $2.0$ so that $\beta=\pm 1$ actually **flips** labels (~40% T0 vs T1 disagreement). With $\lambda_0=2$, $\beta=1$ flipped only ~2% of labels and the interaction was not identifiable.

Model attention logits (query at index; keys are historical events, including injected signals):

$$s_{ij}=\frac{q_i^\top k_j}{\sqrt{d}} - \bigl(\lambda_0+\beta z(a)\bigr)\tau_j$$

The QUERY token is masked as a key so the readout cannot collapse to self-attention at $\tau=0$.


## 3. What each arm can represent

| Arm | Temporal bias | Age input | Can it implement $f(\mathrm{age},\mathrm{lag})$? |
|---|---|---|---|
| `no_age` | none | none | no |
| `temporal_only` | $-\lambda_0\tau$ | none | recency only |
| `late_age` | none | $z(a)$ concatenated at the head | age main effect only |
| `age_temporal` | $-(\lambda_0+\beta z(a))\tau$ | **only** in that bias | **yes** (primary) |

Backbone is matched: 1 layer, $d=64$, 4 heads, BCE, patient-level splits. POS/NEG embeddings are polarity-aligned at init ($+e_0$ / $-e_0$) so the interaction sign is identifiable; the production DKM stack is untouched.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

def find_repo() -> Path:
    here = Path.cwd().resolve()
    for cand in [here, here.parent, Path("/home/suraj/Git/Age-conditioned-pediatric-EHR")]:
        if (cand / "age_temporal_interaction_exp").exists():
            return cand
    raise FileNotFoundError("age_temporal_interaction_exp not found")

REPO = find_repo()
EXP = REPO / "age_temporal_interaction_exp"
RES = EXP / "results"
FIG = RES / "figures"
print("repo:", REPO)
print("results exist:", (RES / "main_results.csv").exists(), "smoke:", (RES / "smoke.json").exists())
print("matched results:", (RES / "matched" / "main_results.csv").exists())


## 4. Synthea cohort, injected signals, planted labels

Reuse the existing sep1-exp pediatric Synthea tables (10,000 patients, ages 0–18, existing train/val/test). Eligible: ≥90 days of pre-index history (**9,407** patients). Synthea events stay as distractors.

Injected events (independent of age):

- 4, 6, or 8 marker events with **balanced** polarities (`n_POS = n_NEG`) so counts cannot label $y$.
- Half of lags $\sim U[7,14]$ days, half $\sim U[60,90]$ days, then shuffled (age-independent).
- One `QUERY` token at index time.


In [ ]:
gen = json.loads((RES / "generator_config.json").read_text())
sanity = json.loads((RES / "sanity.json").read_text())
print("eligible:", gen["n_eligible"], " /", gen["n_total_synthea"])
print("splits:", gen["split_counts"])
print("age groups:", gen["age_group_counts"])
print("tau formula:", gen["tau_formula"])
print("tau_max:", round(gen["tau_max"], 4))
print("age mean/std (train):", round(gen["age_mean_train"], 3), round(gen["age_std_train"], 3))
print("lambda0* / beta*:", gen["lambda0_true"], gen["beta_true"])
print("label agreement T0=T1 / T0=T2 / T1=T2:", sanity.get("label_agreement"))
print("notes:", sanity.get("notes"))


## 5–7. Dataset sanity checks (no training)

We need the age × lag interaction to be **necessary**. Age alone, sequence length, and POS/NEG counts should not solve T1/T2.


In [ ]:
prev = pd.DataFrame(sanity["prevalence_rows"])
show = prev[["task","n","prevalence","corr_age_label","corr_npos_label","corr_nsig_label",
             "prevalence_<1","prevalence_1-5","prevalence_6-11","prevalence_12-17"]].copy()
display(show.round(3))

rows = []
for task, d in sanity["shortcuts"].items():
    rows.append({
        "task": task,
        "age_only_AUROC": d["age_only"]["auroc"],
        "signal_count_AUROC": d["signal_count_only"]["auroc"],
        "npos_minus_nneg_AUROC": d["n_pos_minus_n_neg"]["auroc"],
        "oracle_r_AUROC": d["oracle_r"]["auroc"],
        "oracle_acc": d["oracle_r"]["accuracy"],
    })
display(pd.DataFrame(rows).round(3))
print("sequence length:", json.dumps(sanity.get("sequence_length"), indent=2)[:1200])


In [ ]:
for name, title in [
    ("prevalence_by_age.png", "Label prevalence by age band (should be ~0.5, not age-skewed)"),
    ("signal_lag_by_age.png", "Injected lag distribution (should overlap across ages)"),
    ("lambda_true.png", "Planted λ*(a)"),
    ("kernel_true.png", "Planted temporal kernels b*(a,τ)"),
]:
    fp = FIG / name
    if fp.exists():
        display(Markdown(f"**{title}**"))
        display(Image(filename=str(fp)))


## 8. Tiny overfit smoke test (T1, `age_temporal`, 128 examples, signal-only)

Gate for the full matrix: training BCE / accuracy must collapse, and $\hat\beta$ must move **positive**.


In [ ]:
smoke = json.loads((RES / "smoke.json").read_text())
print("PASS" if smoke["passed"] else "FAIL")
for k, v in smoke.items():
    if k != "lambda_at_ages":
        print(f"  {k}: {v}")
fp = FIG / "smoke_history.png"
if fp.exists():
    display(Image(filename=str(fp)))


## 9–10. Model and training configuration

- 1-layer Transformer, $d_{\mathrm{model}}=64$, 4 heads, FFN 128.
- QUERY hidden state is the patient vector (not mean pooling).
- Loss: BCE with logits. No auxiliary loss. Early stop on validation BCE, patience 6.
- $\lambda_0$ and $\beta$ have no weight decay and 10× learning rate.
- Full matrix: tasks T0/T1/T2 × arms {no_age, temporal_only, late_age, age_temporal} × seeds 0–4.


## 11. Main results table


In [ ]:
main_path = RES / "main_results.csv"
if not main_path.exists():
    display(Markdown("**Full results not written yet.** Re-run this cell after `run_experiment.py --full` finishes."))
else:
    main = pd.read_csv(main_path)
    display(Markdown("Per seed"))
    cols = ["task","model","seed","BCE","AUROC","AUPRC","accuracy","beta_true","beta_hat","lambda0_hat"]
    display(main[cols].round(4))
    agg = pd.read_csv(RES / "main_results_aggregated.csv")
    display(Markdown("Mean ± std across seeds"))
    display(agg.round(4))


## 12. Parameter recovery


In [ ]:
if (RES / "main_results.csv").exists():
    main = pd.read_csv(RES / "main_results.csv")
    rec = main[main["model"]=="age_temporal"][["task","seed","beta_true","beta_hat","lambda0_hat"]].copy()
    display(rec.round(4))
    print(rec.groupby("task")[["beta_hat","lambda0_hat"]].agg(["mean","std"]).round(4))
    if (FIG / "beta_recovery.png").exists():
        display(Image(filename=str(FIG / "beta_recovery.png")))


## 13. $\lambda(a)$ recovery


In [ ]:
fp = FIG / "lambda_recovery.png"
if fp.exists():
    display(Image(filename=str(fp)))
else:
    print("Figure not yet written.")


## 14. Kernel recovery $b(a,\tau)$


In [ ]:
fp = FIG / "kernel_recovery.png"
if fp.exists():
    display(Image(filename=str(fp)))
else:
    print("Figure not yet written.")


## 15. Attention / signal-weight diagnostics

Planted $w^*$ is softmax over **signals only**. Learned attention is softmax over all non-query keys (including Synthea distractors), then renormalized onto signals. Exact equality is not required; treat correlations as diagnostic.


In [ ]:
if (RES / "main_results.csv").exists():
    main = pd.read_csv(RES / "main_results.csv")
    attn = main[main["model"]=="age_temporal"][["task","seed","attn_pearson","attn_spearman","attn_mae","attn_js"]]
    display(attn.round(4))
    print(attn.groupby("task").mean(numeric_only=True).round(4))


## 16. Counterfactual age intervention

Same test sequences; only $z(a)$ changes: correct age, constant (training-mean $\Rightarrow z=0$), shuffled ages.


In [ ]:
ip = RES / "intervention.csv"
if ip.exists():
    inter = pd.read_csv(ip)
    display(inter.round(4))
    display(pd.read_csv(RES / "intervention_aggregated.csv").round(4))
    if (FIG / "intervention.png").exists():
        display(Image(filename=str(FIG / "intervention.png")))


## 17. Comparisons that matter


In [ ]:
if (RES / "main_results.csv").exists():
    main = pd.read_csv(RES / "main_results.csv")
    for metric, fn in [("accuracy","test_accuracy.png"),("BCE","test_bce.png"),("AUROC","test_auroc.png")]:
        fp = FIG / fn
        if fp.exists():
            display(Markdown(f"**Test {metric}**"))
            display(Image(filename=str(fp)))
    print("T1/T2: age_temporal vs temporal_only vs late_age (accuracy mean)")
    sub = main[main["task"].isin(["T1","T2"])]
    print(sub.groupby(["task","model"])["accuracy"].agg(["mean","std"]).round(4))


## 18. Pass / fail summary

Criteria are computed from saved tables (not from memory). Re-run after training to refresh.


In [ ]:
vp = RES / "verdicts.json"
sp = RES / "smoke.json"
lines = []
if sp.exists():
    smoke = json.loads(sp.read_text())
    v1 = "PASS" if smoke["passed"] else "FAIL"
    lines.append(f"""1. Can the minimal model overfit the tiny interaction dataset? **{v1}**
Train BCE={smoke['train_bce']:.3f}, acc={smoke['train_accuracy']:.3f}, β̂={smoke['beta_hat']:+.3f} (true +1) on 128 T1 signal-only examples.""")
if vp.exists() and (RES / "main_results.csv").exists():
    v = json.loads(vp.read_text())
    mapping = [
        ("2_t0_beta_near_zero", "2. When β*=0, does it avoid inventing substantial age dependence?"),
        ("3_t1_beta_positive", "3. When β*>0, does it learn a positive age-temporal relationship?"),
        ("4_t2_beta_negative", "4. When β*<0, does the learned relationship reverse?"),
        ("5_correct_age_helps_only_with_interaction", "5. Does correct age beat shuffled age ONLY when a true interaction exists?"),
        ("6_age_temporal_beats_temporal_only_on_interaction", "6. Does AGE_TEMPORAL_INTERACTION outperform TEMPORAL_ONLY on T1/T2?"),
        ("7_benefit_not_just_late_age", "7. Is the benefit more than an age main effect (vs LATE_AGE)?"),
        ("8_lambda_kernel_resemble_truth", "8. Do learned λ(a) and kernels resemble the planted truth?"),
    ]
    for key, q in mapping:
        item = v[key]
        lines.append(f"""{q} **{item['verdict']}**
{item['detail']}""")
else:
    lines.append("_Full-matrix verdicts appear after `run_experiment.py --full` writes `verdicts.json`._")
display(Markdown("\n\n".join(lines)))


## What this does and does not justify

Ordinary BCE **was** the only training signal. If T1/T2 recover the sign of $\beta$ and shuffled-age hurts only then, the core mechanism works under controlled conditions and there is **not yet** a reason to reintroduce Fourier features, a Chebyshev lag basis, or an auxiliary interaction loss.

If parameter recovery is only sign-correct (not numerically exact), that is expected: content attention and time embeddings are competing recency paths. Exact $\hat\lambda_0=\lambda_0$ is not required.

The original matrix is **not** a strict necessity test: Synthea codes can leak age, so `temporal_only` already reaches high T1/T2 accuracy. Section 19 constructs matched pairs that close that shortcut.

This notebook does **not** move on to MIMIC / pediatric pretraining.


## 19. Matched-pair follow-up: make the age × temporal interaction strictly necessary

The original T1/T2 matrix showed that `age_temporal` recovers the sign of $\beta$ and that shuffling age hurts **only** that arm. It did **not** show that age is required for the labels: `no_age` / `temporal_only` still reached ~0.90 accuracy because Synthea background codes differ across patients.

This follow-up removes that shortcut. Each base history $(x,\tau)$ is copied at two ages, and **only** pairs whose planted labels disagree are kept.

$$
(x,\tau,a_{\mathrm{young}})\mapsto y_{\mathrm{young}},\qquad
(x,\tau,a_{\mathrm{old}})\mapsto y_{\mathrm{old}},\qquad
y_{\mathrm{young}}\neq y_{\mathrm{old}}
$$

with the same planted rule $b(a,\tau)=-(\lambda_0+\beta z(a))\tau$, $\lambda_0=0.5$, $a_{\mathrm{young}}=2$, $a_{\mathrm{old}}=16$, $z=\pm 1$. Signal injection is still age-independent. Splits are by **base history**, so the two copies of a pair cannot leak across train/test.

| | |
|---|---|
| Code | `age_temporal_interaction_exp/run_matched_experiment.py` |
| Raw tables | `age_temporal_interaction_exp/results/matched/*.csv` / `*.json` |
| Figures | `age_temporal_interaction_exp/results/matched/figures/` |
| Previous results | **unchanged** under `results/main_results.csv` |


In [ ]:
MRES = EXP / "results" / "matched"
MFIG = MRES / "figures"
print("matched results dir:", MRES)
print("exists:", MRES.exists())
if (MRES / "generator_config.json").exists():
    mgen = json.loads((MRES / "generator_config.json").read_text())
    display(Markdown(
        "**Construction.** {n} flipping pairs × 2 ages = {e} examples. "
        "Train/val/test pairs = {sp}. Ages {ay:.0f}y and {ao:.0f}y ($z=\\pm 1$). "
        "Margin $|r|>{m}$. Balanced age→label directions.".format(
            n=mgen["n_pairs"], e=mgen["n_examples"], sp=mgen["split_pairs"],
            ay=mgen["age_young"], ao=mgen["age_old"], m=mgen["margin"],
        )
    ))
    display(mgen)
if (MRES / "sanity.json").exists():
    msan = json.loads((MRES / "sanity.json").read_text())
    display(Markdown("**Pair checks** (must be 1.0 / 1.0 / 1.0)"))
    display(msan["pair_checks"])
    display(Markdown("**Shortcut AUROCs** (must be chance: age and POS/NEG counts cannot label a pair)"))
    display(pd.DataFrame(msan["shortcuts"]).T)


### 19.1 What each arm can use

Events, timestamps, and lags are **identical** within a pair. A model that ignores age must emit the same prediction for both members, so it cannot classify both correctly. Ordinary accuracy can still look like 0.5; **pair accuracy** (both members correct) is the quantity that exposes that failure.

`late_age` receives $z(a)$ only as an additive feature at the linear head. It may still pick up a global age shift, but a single additive offset cannot implement *history-dependent* flip directions (half the pairs are young→1/old→0, half the reverse). We report whatever it actually achieves.

Same four arms, same 1-layer architecture, ordinary BCE only, 5 seeds, tasks T1 ($\beta^*=+1$) and T2 ($\beta^*=-1$).


In [ ]:
mp = MRES / "main_results.csv"
if not mp.exists():
    display(Markdown("**Matched-pair results not written yet.** Re-run this cell after `run_matched_experiment.py --full` finishes."))
else:
    mm = pd.read_csv(mp)
    display(Markdown("Per seed"))
    cols = ["task","model","seed","BCE","AUROC","AUPRC","accuracy","pair_accuracy","same_prediction_rate","beta_hat","lambda0_hat","delta_shuffle_mean"]
    display(mm[cols].round(4))
    display(Markdown("Mean ± std across seeds"))
    display(pd.read_csv(MRES / "main_results_aggregated.csv").round(4))


### 19.2 Matched-pair accuracy

For each test pair, the model is correct only if it classifies **both** the young and old copies. `same_prediction_rate` is the fraction of pairs where the model predicts the same label at both ages (expected ≈ 1 for `no_age` / `temporal_only`).


In [ ]:
if (MRES / "main_results.csv").exists():
    mm = pd.read_csv(MRES / "main_results.csv")
    g = mm.groupby(["task","model"])[["accuracy","pair_accuracy","same_prediction_rate","AUROC"]].agg(["mean","std"])
    display(g.round(4))
    for fn, cap in [
        ("matched_accuracy.png", "Test accuracy"),
        ("matched_pair_accuracy.png", "Matched-pair accuracy"),
        ("matched_auroc.png", "Test AUROC"),
    ]:
        fp = MFIG / fn
        if fp.exists():
            display(Markdown(f"**{cap}**"))
            display(Image(filename=str(fp)))


### 19.3 $\hat\beta$ and age shuffle


In [ ]:
if (MRES / "main_results.csv").exists():
    mm = pd.read_csv(MRES / "main_results.csv")
    rec = mm[mm["model"]=="age_temporal"][["task","seed","beta_true","beta_hat","lambda0_hat","delta_shuffle_mean","pair_accuracy","pair_accuracy_shuffled_age"]]
    display(rec.round(4))
    print(rec.groupby("task")[["beta_hat","lambda0_hat","delta_shuffle_mean","pair_accuracy","pair_accuracy_shuffled_age"]].agg(["mean","std"]).round(4))
    if (MRES / "intervention.csv").exists():
        display(Markdown("Correct-age vs shuffled-age BCE"))
        display(pd.read_csv(MRES / "intervention.csv").round(4))
    for fn in ("matched_beta.png", "matched_shuffle.png"):
        fp = MFIG / fn
        if fp.exists():
            display(Image(filename=str(fp)))


### 19.4 Verdicts


In [ ]:
mvp = MRES / "verdicts.json"
msp = MRES / "smoke.json"
lines = []
if msp.exists():
    sm = json.loads(msp.read_text())
    v = "PASS" if sm["passed"] else "FAIL"
    lines.append(
        "**Smoke (64 pairs).** **{v}** train BCE={bce:.3f}, acc={acc:.3f}, "
        "pair_acc={pa:.3f}, β̂={beta:+.3f}.".format(
            v=v, bce=sm["train_bce"], acc=sm["train_accuracy"],
            pa=sm.get("train_pair_accuracy", float("nan")), beta=sm["beta_hat"],
        )
    )
if mvp.exists():
    vv = json.loads(mvp.read_text())
    mapping = [
        ("1_interaction_necessary", "1. Is the age × temporal interaction necessary in this dataset?"),
        ("2_age_temporal_solves", "2. Can the simple age_temporal mechanism learn it?"),
        ("3_temporal_only_cannot", "3. Can temporal_only solve it without age?"),
        ("4_late_age_indirect", "4. Can late_age learn the interaction indirectly?"),
        ("5_beta_sign", "β̂ sign (planted direction)"),
        ("6_shuffle_breaks_age_temporal", "5. Does shuffling age break age_temporal as expected?"),
        ("7_beats_temporal_only", "Does age_temporal outperform temporal_only?"),
    ]
    for key, q in mapping:
        item = vv[key]
        lines.append("{q} **{ver}**\n{detail}".format(q=q, ver=item["verdict"], detail=item["detail"]))
else:
    lines.append("_Matched-pair verdicts appear after `run_matched_experiment.py --full` writes `results/matched/verdicts.json`._")
display(Markdown("\n\n".join(lines)))
